<a href="https://colab.research.google.com/github/rezzz59/Sentimen-Analysis-Aplikasi-Grab/blob/main/grab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
%pip install google-play-scrapper

In [12]:
import pandas as pd
from google_play_scraper import Sort, reviews

result, countinuation_token = reviews(
    'com.grabtaxi.passenger',
    lang = 'id',
    country = 'id',
    sort = Sort.NEWEST,
    count= 11000,
)

df = pd.DataFrame(result)

df = df[['content', 'score']]

df.to_csv('grab_review_raw.csv', index = False)

In [13]:
df

,content,score
0,sangat puas,5
1,sopir minta batal tetapi saldo terpotong dan l...,1
2,Semangkin lama semangkin tak logika. perjalana...,2
3,baik,5
4,Aplikasi sampah,1
...,...,...
10995,aplikasi terbaik,5
10996,mantap,5
10997,Pelayanannya lama. sering order go food malam ...,1
10998,limit ovo paylater malah turun jadi 1 perak,1


In [23]:
import re
import string

slang_dict = {
    "ga": "tidak", "gak": "tidak", "gakk": "tidak", "nggak": "tidak",
    "yg": "yang", "dr": "dari", "bgt": "banget", "kl": "kalau",
    "udh": "sudah", "udah": "sudah", "aja": "saja", "jd": "jadi",
    "tp": "tapi", "pake": "pakai", "sdh": "sudah", "aja": "saja",
    "dapet": "dapat", "ilang": "hilang", "lemot": "lambat", "gercep": "cepat",
}

def clean_text(text):
  text = text.lower()

  #menghapus link, mention dan tag
  text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
  text = text = re.sub(r'@\w+|\#', '', text)

  #menghapus angka dan tanda bca
  text = text.translate(str.maketrans('', '', string.punctuation))
  text = re.sub(r'\d+', '', text)

  #menghapus emoji
  text = text.encode('ascii', 'ignore').decode('ascii')

  #normalisasi kata & tokenizing
  words = text.split()
  cleaned_words = [slang_dict.get(w, w) for w in words]

  return " ".join(cleaned_words)

df['content_cleaned'] = df['content'].apply(clean_text)
df = df[df['content_cleaned'] != '']


In [27]:
def labeling(score):
  if score > 3:
    return "positif"
  if score < 2:
    return 'negatif'
  else:
    return 'netral'

df['label'] = df['score'].apply(labeling)

/tmp/ipykernel_16331/1599349615.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['score'].apply(labeling)


In [30]:
df

,content,score,content_cleaned,label
0,sangat puas,5,sangat puas,positif
1,sopir minta batal tetapi saldo terpotong dan l...,1,sopir minta batal tetapi saldo terpotong dan l...,negatif
2,Semangkin lama semangkin tak logika. perjalana...,2,semangkin lama semangkin tak logika perjalanan...,netral
3,baik,5,baik,positif
4,Aplikasi sampah,1,aplikasi sampah,negatif
...,...,...,...,...
10995,aplikasi terbaik,5,aplikasi terbaik,positif
10996,mantap,5,mantap,positif
10997,Pelayanannya lama. sering order go food malam ...,1,pelayanannya lama sering order go food malam m...,negatif
10998,limit ovo paylater malah turun jadi 1 perak,1,limit ovo paylater malah turun jadi perak,negatif


In [31]:
df[['content_cleaned', 'score', 'label']]

,content_cleaned,score,label
0,sangat puas,5,positif
1,sopir minta batal tetapi saldo terpotong dan l...,1,negatif
2,semangkin lama semangkin tak logika perjalanan...,2,netral
3,baik,5,positif
4,aplikasi sampah,1,negatif
...,...,...,...
10995,aplikasi terbaik,5,positif
10996,mantap,5,positif
10997,pelayanannya lama sering order go food malam m...,1,negatif
10998,limit ovo paylater malah turun jadi perak,1,negatif
